## Importing Libraries

In [1]:
!pip install deepxde

In [ ]:
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt

### Defining Heat Equation

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

PINN residual (should be zero in the interior):

$$\frac{\partial u}{\partial t} - \alpha \frac{\partial^2 u}{\partial x^2} = 0$$

- **Space:** $x \in [-1, 1]$
- **Time:** $t \in [0, 1]$
- **Diffusivity:** $\alpha$ (set below)

In [3]:
HEAT_COEFFICIENT = 0.01
STARTING_TIME = 0.0
ENDING_TIME = 1.0
STARTING_POSITION = 0.0
ENDING_POSITION = 1.0

In [4]:
def heat_equation(x, u, v):
    du_t = dde.grad.jacobian(u, x, i=0, j=1)
    du_xx = dde.grad.hessian(u, x, i=0, j=0)
    return du_t - (HEAT_COEFFICIENT * du_xx)

### Definint Time and Space Interval

- *Time Interval* `0` to `1`
- *Space Interval* `0` to `1`

In [5]:
time = dde.geometry.TimeDomain(STARTING_TIME, ENDING_TIME)
space = dde.geometry.Interval(STARTING_POSITION, ENDING_POSITION)
geometry = dde.geometry.GeometryXTime(space, time)

### Initial and Boundary Condition

In [6]:
def boundary_value(x):
    return np.zeros((len(x), 1))   # u=0 at x=0 and x=1

def is_boundary(_, on_boundary):
    return on_boundary

def is_initial(_, on_initial):
    return on_initial

def initial_condition(x):
    return np.sin(np.pi * x[:, 0:1])  # u(x,0) = sin(pi*x)


bc = dde.icbc.DirichletBC(geometry, boundary_value, is_boundary)
ic = dde.icbc.IC(geometry, initial_condition, is_initial)  # ← fixed

### Defining Data

In [7]:
sensor_x = np.linspace(STARTING_POSITION, ENDING_POSITION, 200)[:, None]
space = dde.data.GRF((ENDING_POSITION - STARTING_POSITION), length_scale=0.2, N=1000, interp="cubic")

In [8]:
time_data = dde.data.TimePDE(
    geometry,
    heat_equation,
    [bc, ic],
    num_domain=100,
    num_boundary=20,
    num_initial=20,
)

data = dde.data.PDEOperatorCartesianProd(
    time_data,
    space,
    sensor_x,
    num_function=200,
    function_variables=[0],
    num_test=100
)

In [ ]:
p   = 64
net = dde.nn.DeepONetCartesianProd(
    [200, 40, 40, p],
    [2,   40, 40, p],
    "tanh", "Glorot normal"
)

model = dde.Model(data, net)

model.compile("adam", lr=0.001)
losshistory, train_state = model.train(iterations=100)

model.compile("L-BFGS")
losshistory, train_state = model.train()

Compiling model...
'compile' took 0.008939 s

Training model...



KeyboardInterrupt: 